## Notebook 04 — Cross-Dataset Comparison Visualizations

Aggregates results from Notebooks 01–03 and generates the comparison figures.  
**Requires:** Checkpoints produced by Notebooks 01, 02, and 03 to be present in `BASE_DIR`.  

> Run the **Configuration** cell below before any other cell to set data paths.


In [ ]:
from pathlib import Path

# ── DATA PATHS — adjust to your local setup ──────────────────────────────────
# Point BASE_DIR to the folder that contains your checkpoint sub-directories.
# Expected structure:
#   BASE_DIR/
#     edgeiiot_checkpoints/   <- Edge-IIoT intermediate results
#     cicids2017_checkpoints/ <- CIC-IDS-2017 intermediate results
#     dnn_lstm_checkpoints/   <- DNN / LSTM intermediate results
#     ML-EdgeIIoT-dataset.csv <- raw Edge-IIoT CSV
BASE_DIR       = Path("../data")
EDGEIIOT_DIR   = BASE_DIR / "edgeiiot_checkpoints"
CICIDS2017_DIR = BASE_DIR / "cicids2017_checkpoints"
DNN_LSTM_DIR   = BASE_DIR / "dnn_lstm_checkpoints"
FIGURES_DIR    = Path("../figures")
RESULTS_DIR    = BASE_DIR / "analiz_sonuclari"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import pickle, os

EDGE_DIR = r"str(EDGEIIOT_DIR)"
CIC_DIR  = r"str(CICIDS2017_DIR)"

def yukle_dir(dizin, dosya):
    with open(os.path.join(dizin, dosya), 'rb') as f:
        return pickle.load(f)

# Edge-IIoT
df_xai_edge  = yukle_dir(EDGE_DIR, "df_xai_score.pkl")
df_var_edge  = yukle_dir(EDGE_DIR, "df_var.pkl")
df_fid_edge  = yukle_dir(EDGE_DIR, "df_fidelity.pkl")
perturb_edge = yukle_dir(EDGE_DIR, "tum_perturb_sonuclar.pkl")

# CIC-IDS2017
df_xai_cic   = yukle_dir(CIC_DIR, "df_xai_score_cicids.pkl")
df_var_cic   = yukle_dir(CIC_DIR, "df_var_cicids.pkl")
df_fid_cic   = yukle_dir(CIC_DIR, "df_fidelity_cicids.pkl")
perturb_cic  = yukle_dir(CIC_DIR, "tum_perturb_sonuclar_cicids.pkl")

# Yapıları kontrol et
print("=== df_var_edge ===")
print(df_var_edge.head(3))
print("\n=== df_fid_edge ===")
print(df_fid_edge.head(3))
print("\n=== df_var_cic ===")
print(df_var_cic.head(3))
print("\n=== df_fid_cic ===")
print(df_fid_cic.head(3))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

EDGE_DIR  = r"str(EDGEIIOT_DIR)"
CIC_DIR   = r"str(CICIDS2017_DIR)"
GORSEL_DIR = r"str(CICIDS2017_DIR)"

def yukle_dir(dizin, dosya):
    with open(os.path.join(dizin, dosya), 'rb') as f:
        return pickle.load(f)

RENKLER = {
    'RandomForest':'#2196F3','ExtraTrees':'#4CAF50','DecisionTree':'#FF9800',
    'XGBoost':'#F44336','LightGBM':'#9C27B0','CatBoost':'#00BCD4',
    'LogisticRegression':'#795548'
}
KISALTMA = {
    'RandomForest':'RF','ExtraTrees':'ET','DecisionTree':'DT',
    'XGBoost':'XGB','LightGBM':'LGBM','CatBoost':'CB','LogisticRegression':'LR'
}
MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# ── Verileri yükle ───────────────────────────────────────────────────
df_xai_edge  = yukle_dir(EDGE_DIR, "df_xai_score.pkl")
df_var_edge  = yukle_dir(EDGE_DIR, "df_var.pkl")
df_fid_edge  = yukle_dir(EDGE_DIR, "df_fidelity.pkl")
perturb_edge = yukle_dir(EDGE_DIR, "tum_perturb_sonuclar.pkl")

df_xai_cic   = yukle_dir(CIC_DIR, "df_xai_score_cicids.pkl")
df_var_cic   = yukle_dir(CIC_DIR, "df_var_cicids.pkl")
df_fid_cic   = yukle_dir(CIC_DIR, "df_fidelity_cicids.pkl")
perturb_cic  = yukle_dir(CIC_DIR, "tum_perturb_sonuclar_cicids.pkl")

# ── Metrikleri çıkar ─────────────────────────────────────────────────
# RQ1: SHAP @5% pertürbasyon
def get_perturb_shap5(perturb):
    return {m: np.nanmean(perturb[m]['pct5']['shap']['spearman']) for m in MODEL_SIRASI}

rq1_edge = get_perturb_shap5(perturb_edge)
rq1_cic  = get_perturb_shap5(perturb_cic)

# RQ2: Ortalama Spearman (model varyasyonu)
rq2_edge = df_var_edge.groupby('model_adi')['spearman'].mean().to_dict()
rq2_cic  = df_var_cic.groupby('model_adi')['spearman'].mean().to_dict()

# Fidelity: SHAP @top5 delta_f1
def get_fidelity5(df_fid):
    df5 = df_fid[(df_fid['xai']=='SHAP') & (df_fid['n_maskele']==5)]
    return df5.groupby('model')['delta_f1'].mean().to_dict()

fid_edge = get_fidelity5(df_fid_edge)
fid_cic  = get_fidelity5(df_fid_cic)

# XAI-Score
xai_edge = df_xai_edge.set_index('Model')['XAI_Score'].to_dict()
xai_cic  = df_xai_cic.set_index('Model')['XAI_Score'].to_dict()

# ── Grafik ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Edge-IIoT vs CIC-IDS-2017 — XAI Güvenilirlik Karşılaştırması',
             fontsize=15, fontweight='bold', y=1.01)

metr_listesi = [
    (axes[0,0], rq1_edge,  rq1_cic,  'RQ1: Pertürbasyon Kararlılığı (SHAP @%5)',  'Spearman r'),
    (axes[0,1], rq2_edge,  rq2_cic,  'RQ2: Model Varyasyonu (SHAP Spearman)',     'Spearman r'),
    (axes[1,0], fid_edge,  fid_cic,  'Fidelity: SHAP @Top-5 Özellik (ΔF1)',      'ΔF1'),
    (axes[1,1], xai_edge,  xai_cic,  'XAI Score (Genel Güvenilirlik)',            'XAI Score'),
]

x      = np.arange(len(MODEL_SIRASI))
bar_w  = 0.35

for ax, edge_d, cic_d, baslik, ylabel in metr_listesi:
    vals_edge = [edge_d.get(m, 0) for m in MODEL_SIRASI]
    vals_cic  = [cic_d.get(m, 0)  for m in MODEL_SIRASI]
    renkler   = [RENKLER[m] for m in MODEL_SIRASI]

    b1 = ax.bar(x - bar_w/2, vals_edge, bar_w,
                color=renkler, alpha=1.0, edgecolor='white', linewidth=0.8, label='Edge-IIoT')
    b2 = ax.bar(x + bar_w/2, vals_cic,  bar_w,
                color=renkler, alpha=0.45, edgecolor='grey', linewidth=0.8,
                hatch='//', label='CIC-IDS-2017')

    for bar, v in zip(b1, vals_edge):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    for bar, v in zip(b2, vals_cic):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7, color='#444')

    ax.set_xticks(x)
    ax.set_xticklabels([KISALTMA[m] for m in MODEL_SIRASI], fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(baslik, fontsize=12, fontweight='bold')
    ax.set_ylim(0, max(max(vals_edge), max(vals_cic)) * 1.2)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

    # Legend sadece ilk grafike
    if ax == axes[0,0]:
        leg_patches = [
            mpatches.Patch(facecolor='grey', alpha=1.0, label='Edge-IIoT (düz)'),
            mpatches.Patch(facecolor='grey', alpha=0.45, hatch='//', label='CIC-IDS-2017 (taralı)'),
        ]
        ax.legend(handles=leg_patches, fontsize=9, loc='lower right')

plt.tight_layout()
kayit_yolu = os.path.join(GORSEL_DIR, 'karsilastirma_edge_vs_cicids.png')
plt.savefig(kayit_yolu, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {kayit_yolu}")